#PART A: Data Processing

In [ ]:
# mount google drive
from google.colab import drive
drive.mount("/content/gdrive/")

In [ ]:
dataset_path = "/content/gdrive/MyDrive/school/APS360 - Deep Learning/APS360 shared drive/dataset/cropped_dataset.zip"

In [ ]:
# extract the .zip file to /content/project/dog_breed_dataset of local colab env
import zipfile
import os

data_dir = '/content/project/dog_breed_dataset'

os.makedirs(data_dir, exist_ok=True)

zip_ref = zipfile.ZipFile(dataset_path, 'r')
zip_ref.extractall(data_dir)
zip_ref.close()

In [ ]:
# split dataset
!pip install split-folders
import splitfolders

save_dir = "/content/project/dog_breed_dataset/content/project/cropped_dataset/pictures" # debug
input_dir = save_dir
output_dir = "/content/project/output_split"
!mkdir {output_dir}

splitfolders.ratio(
    input=input_dir,
    output=output_dir,
    seed=1,
    ratio=(0.7, 0.20, 0.10),  # 70% train, 20% val, 10% test
    group_prefix=None
)

In [ ]:
!rm -rf "/content/project/output_split/train/.ipynb_checkpoints"
!rm -rf "/content/project/output_split/val/.ipynb_checkpoints"
!rm -rf "/content/project/output_split/test/.ipynb_checkpoints"

In [ ]:
# data augmentation: to help with model performance

from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((300, 300)), # can change, having fixed size is good for the FC layers
    transforms.RandomCrop((280, 280)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(), # convert .jpg to PyTorch tensors
    transforms.Normalize([0.485, 0.456, 0.406], # normalize with ImageNet mean
                         [0.229, 0.224, 0.225]) # and standard deviation
])

val_and_test_transforms = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.CenterCrop((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
# data loading: converts .jpg to tensors

from torchvision import datasets
from torch.utils.data import DataLoader, Subset


def get_data_loader(batch_size):
    # paths
    root_dir = "/content/project/output_split"
    train_dir = os.path.join(root_dir, "train")
    val_dir = os.path.join(root_dir, "val")
    test_dir = os.path.join(root_dir, "test")

    # create train, val, and test datasets
    train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
    val_dataset = datasets.ImageFolder(root=val_dir, transform=val_and_test_transforms)
    test_dataset = datasets.ImageFolder(root=test_dir, transform=val_and_test_transforms)

    classes = train_dataset.classes

    # create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader, classes

#Part B: Baseline Model

It is important to ensure that the performance and accuracy of a model is not just achieved through
the complexity of the architecture, but through meaningful iterations and effective tuning of hyper-
parameters. The team decided to utilize both a Support Vector Machine (SVM) and an ANN Model
to comprehensively quantify the model’s accuracy

###Helper Functions

In [ ]:
'''Library Imports'''
#ML
import numpy as np
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.utils.data.sampler import SubsetRandomSampler
import torchvision.transforms as transforms

#Plot
import os
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

#Os
import os
import shutil
import random

#SkLearn (For SVM)
import numpy as np
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.multiclass import OneVsOneClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample  # For random sampling

#Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
###############################################################################
# Training
def get_model_name(name, batch_size, learning_rate, epoch):
    """ Generate a name for the model consisting of all the hyperparameter values

    Args:
        config: Configuration object containing the hyperparameters
    Returns:
        path: A string with the hyperparameter name and value concatenated
    """
    path = "model_{0}_bs{1}_lr{2}_epoch{3}".format(name,
                                                   batch_size,
                                                   learning_rate,
                                                   epoch)
    return path

def evaluate(net, loader, criterion):
    """Evaluate the network on the validation set.

    Args:
        net: PyTorch neural network object
        loader: PyTorch data loader for the validation set
        criterion: The loss function
    Returns:
        err: A scalar for the average classification error over the validation set
        loss: A scalar for the average loss function over the validation set
    """
    net.eval()  # Set the model to evaluation mode
    total_loss = 0.0
    total_err = 0.0
    total_epoch = 0

    with torch.no_grad():  # Disable gradient calculation for evaluation
        for i, data in enumerate(loader, 0):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, labels)  # No normalization now b/c cross entropy
            _, predicted = torch.max(outputs, 1)
            total_err += (predicted != labels).sum().item()  # Count misclassifications
            total_loss += loss.item()
            total_epoch += len(labels)
    err = float(total_err) / total_epoch
    loss = float(total_loss) / (i + 1)
    return err, loss


###############################################################################
# Training Curve
def plot_training_curve(path):
    """ Plots the training curve for a model run, given the csv files
    containing the train/validation error/loss.

    Args:
        path: The base path of the csv files produced during training
    """
    train_err = np.loadtxt("{}_train_err.csv".format(path))
    val_err = np.loadtxt("{}_val_err.csv".format(path))
    train_loss = np.loadtxt("{}_train_loss.csv".format(path))
    val_loss = np.loadtxt("{}_val_loss.csv".format(path))
    plt.title("Train vs Validation Error")
    n = len(train_err) # number of epochs, must be > 1
    plt.plot(range(1,n+1), train_err, label="Train")
    plt.plot(range(1,n+1), val_err, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Error")
    plt.legend(loc='best')
    plt.show()
    plt.title("Train vs Validation Loss")
    plt.plot(range(1,n+1), train_loss, label="Train")
    plt.plot(range(1,n+1), val_loss, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(loc='best')
    plt.show()

###Trainer

In [ ]:
def train_net(net, batch_size=64, learning_rate=0.01, num_epochs=1):  #Modified from Lab2 to include multi-class classification
    # Define device here:
    ########################################################################
    # Fixed PyTorch random seed for reproducible result
    torch.manual_seed(1000)
    ########################################################################
    # Move the model to the appropriate device
    net.to(device)

    # Obtain the PyTorch data loader objects to load batches of the datasets
    train_loader, val_loader, test_loader, classes = get_data_loader(batch_size)
    ########################################################################
    # Define the Loss function and optimizer and scheduler
    criterion = nn.CrossEntropyLoss()  # multiclass classification
    # optimizer = optim.SGD(net.parameters(), lr=learning_rate, momentum=0.9)
    optimizer = optim.Adam(net.parameters(), lr=learning_rate, weight_decay=1e-5)
    # scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    ########################################################################
    # Set up some numpy arrays to store the training/test loss/accuracy
    train_err = np.zeros(num_epochs)
    train_loss = np.zeros(num_epochs)
    val_err = np.zeros(num_epochs)
    val_loss = np.zeros(num_epochs)
    ########################################################################
    # Train the network
    # Loop over the data iterator and sample a new batch of training data
    # Get the output from the network, and optimize our loss function.
    start_time = time.time()
    for epoch in range(num_epochs):  # loop over the dataset multiple times
        total_train_loss = 0.0
        total_train_err = 0.0
        total_epoch = 0
        # scheduler.step()

        for i, data in enumerate(train_loader, 0):
            # Get the inputs
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)  # Move data to the appropriate device
            # Zero the parameter gradients
            optimizer.zero_grad()
            # Forward pass, backward pass, and optimize
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            # Calculate the statistics
            null, predicted = torch.max(outputs, 1)
            corr = (predicted != labels).sum().item()
            total_train_err += corr
            total_train_loss += loss.item()
            total_epoch += len(labels)
        train_err[epoch] = float(total_train_err) / total_epoch
        train_loss[epoch] = float(total_train_loss) / (i + 1)
        val_err[epoch], val_loss[epoch] = evaluate(net, val_loader, criterion)

        print(("Epoch {}: Train err: {}, Train loss: {} |" +
                "Validation err: {}, Validation loss: {}").format(
                    epoch + 1,
                    train_err[epoch],
                    train_loss[epoch],
                    val_err[epoch],
                    val_loss[epoch]))

        # Save the current model (checkpoint) to a file
        model_path = get_model_name(net.name, batch_size, learning_rate, epoch+1)
        torch.save(net.state_dict(), model_path)


    print('Finished Training')
    end_time = time.time()
    elapsed_time = end_time - start_time
    print("Total time elapsed: {:.2f} seconds".format(elapsed_time))

    # Write the train/test loss/err into CSV file for plotting later
    epochs = np.arange(1, num_epochs + 1)
    np.savetxt("{}_train_err.csv".format(model_path), train_err)
    np.savetxt("{}_train_loss.csv".format(model_path), train_loss)
    np.savetxt("{}_val_err.csv".format(model_path), val_err)
    np.savetxt("{}_val_loss.csv".format(model_path), val_loss)

#Part C: Primary Model

In [ ]:
# RizzNet v2

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # downsample layer to match dimensions for skip connection
        self.downsample = nn.Identity()
        if in_channels != out_channels:
            self.downsample = nn.Sequential(
                # the kernel = 1 is equivalent to padding = 1 and stride = 2
                # since the formula is (o+2p-k)/2 + 1
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.downsample(x)
        x = F.leaky_relu(self.bn1(self.conv1(x)), negative_slope=0.2)
        x = self.bn2(self.conv2(x))
        x += identity
        x = F.leaky_relu(x, negative_slope=0.2)
        return x

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.name = "CNN"

        self.conv1 = nn.Conv2d(3, 64, 5, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        # output 64 x 75 x 75

        self.res_block1 = ResidualBlock(64, 128, stride=2)
        self.res_block1 = ResidualBlock(128, 128, stride=1)
        self.res_block2 = ResidualBlock(128, 256, stride=2)
        self.res_block3 = ResidualBlock(256, 512, stride=2)
        self.res_block4 = ResidualBlock(512, 512, stride=1)

        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(512, 120)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(F.leaky_relu(self.bn1(self.conv1(x)), negative_slope=0.2))

        x = self.dropout(self.res_block1(x))
        x = self.dropout(self.res_block2(x))
        x = self.dropout(self.res_block3(x))
        x = self.dropout(self.res_block4(x))
        x = self.dropout(self.res_block5(x))

        # Global pooling
        # reshapes the tensor from (batch_size, channels, 1, 1) to (batch_size, channels).
        x = self.global_avg_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc1(x)

        return x

# hyperparameters
batch_size = 64
learning_rate = 0.001
num_epochs = 80

# train
net = CNN()
train_net(net=net, batch_size=batch_size, num_epochs=num_epochs, learning_rate=learning_rate)
cnnNetPath = get_model_name("CNN", batch_size=batch_size, learning_rate=learning_rate, epoch=num_epochs)
plot_training_curve(cnnNetPath)